# agentfix (ReAct edition) — Google Colab

Browser-only setup. No local install, no 8 GB download.

This is the agent that **reasons before it acts**. Run the cells **in order**;
sections 0–6 are setup, and everything after that is the agent actually working.

The one thing to watch for: a `thinks` line under each turn of the trace. The previous
edition of this workshop could not produce one.


## 0. Check the runtime

A CPU runtime is fine — the model is small. No GPU needed.


In [ ]:
!python --version
!free -g | head -2 || true


## 1. Configuration

Change `REPO_URL` if you forked the repository.

`MODEL` matters more than usual here. It has to be a model that **reasons** — a free Colab
runtime cannot hold Mellum2-Thinking, and the obvious small substitute (`qwen2.5-coder:1.5b`,
which the previous edition used) has no thinking mode at all. Point this notebook at that and
everything still runs, silently, as the *previous* workshop's Act-only agent. `qwen3:1.7b` is
the smallest thing that both thinks and calls tools.


In [ ]:
REPO_URL = "https://github.com/jelenadjuric01/agentfix-react.git"
MODEL = "qwen3:1.7b"   # thinks AND calls tools. ~1.4 GB.
print(REPO_URL, MODEL)


## 2. Install and start Ollama

The server runs in the background inside this runtime and listens on `localhost:11434`.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!curl -s http://localhost:11434/api/tags | head -c 200


## 3. Pull the model

No `ollama create` step is needed. The client talks to Ollama's native API, which honours a
per-request `num_ctx`, so the context window is set by the client rather than baked into a
derived model. (The `Modelfile` in the repo is for the local Mellum2 path.)


In [ ]:
import os; os.environ["MODEL"]=MODEL
!ollama pull $MODEL


## 4. Clone the repository


In [ ]:
%cd /content
!rm -rf agentfix-react
!git clone --quiet $REPO_URL agentfix-react
%cd /content/agentfix-react
!git branch --show-current


## 5. Make accidental pushes impossible

You will be committing nothing and pushing nothing, but a stray `git push` in a shared
notebook is worth ruling out.


In [ ]:
!git remote set-url --push origin DISABLED
!git remote -v


## 6. Install the project and check the setup

Colab already has a Python, so this installs into it directly rather than via `uv`.


In [ ]:
!python -m pip install -q -e '.[dev]'
import os; os.environ["MELLUM_MODEL"] = MODEL
!agentfix doctor || true


Read two lines in particular:

- **`reasoning`** — proof the model thinks, and that the thinking arrives on its own channel
  rather than inline as `<think>` tags. If this FAILs, nothing below is the ReAct agent.
- **`tool calling`** — proof it can still act while thinking. A model that reasons beautifully
  and calls nothing changes no files.

`ram` may report differently here than on a laptop, and the model name comes from
`MELLUM_MODEL`, which the cell above set for you.


# The agent, on a real bug

`tasks/workshop/01-shopcart` is a small project with a failing test. Nothing about the bug is
given to the agent — it has to run the tests, read the failure, find the cause and verify a fix.


In [ ]:
!agentfix solve tasks/workshop/01-shopcart --verbose


## What to read in that trace

One line per model turn, one per tool call, and the context size growing across the run.

The new part is the indented **`thinks`** line under each turn. That is the model's reasoning,
arriving on a separate channel from its answer — which is why `write_file` still receives a
clean file and not a monologue.

Three things worth looking for specifically:

1. **The summary line reports `thinks=N/N`.** The previous edition's measured result was
   `0 of 7`, and closing that gap is what this edition is.
2. **Reasoning is not free.** Compare `tokens` and the peak context against the previous
   edition's numbers in the README. Thinking is generated tokens, and it is re-sent on every
   later turn.
3. **A wrong thought is still wrong.** Watch for a turn where the model reasons its way
   confidently to the wrong file. The tests are what catch it — not the argument.


## The harder task

Task 02's bug is **not** in the file its failing test names, which is why `list_files` and
`read_file` earn their place. A 1.7B model may not solve it; read the reasoning either way.


In [ ]:
!agentfix solve tasks/workshop/02-invoice --verbose


# The test suite

Runs with no model process anywhere. `llm/fake.py` is a real chat model that returns a
scripted list of replies — including replies that carry reasoning, in exactly the field the
real client uses — so the tests drive the **real** graph against the **real** tools in a real
temp directory. Only the model is replaced.


In [ ]:
!python -m unittest discover -s tests -t . 2>&1 | tail -5


In [ ]:
# The reasoning behaviour specifically: what makes this the ReAct edition.
!python -m unittest tests.test_reasoning -v 2>&1 | tail -30


## Optional — what the framework does and does not give you

`agent/prebuilt.py` builds the same agent from `langchain.agents.create_agent` and its
middleware, and documents which of the invariants that actually buys — including the one
reasoning made newly relevant: there is no seam for a thinking guard.


In [ ]:
!python -m pip install -q -e '.[dev,prebuilt]'
!python -m unittest discover -s tests -p 'test_prebuilt.py' -t . -v 2>&1 | tail -20


## Optional — an eval run

Slow: it runs the model once per task, sequentially, and that is deliberate — one local model
is one set of weights being time-shared. Precomputed results from a Mellum2-Thinking machine
are in `results/precomputed/` if you would rather read than wait.


In [ ]:
!cat results/precomputed/workshop.json
# !agentfix eval --suite workshop     # uncomment to run it for real
